In [21]:
from shutil import copytree
from time import perf_counter
from typing import Any, Generator

import coremltools as ct
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizer
from transformers.cache_utils import Cache

In [9]:
class SliceUpdateKeyValueCache(Cache):
    def __init__(
        self,
        shape: tuple[int, ...],
        device="cpu",
        dtype=torch.float32,
    ) -> None:
        """KV cache of shape (#layers, batch_size, #kv_heads, context_size, head_dim)."""
        super().__init__(layers=[])
        self.past_seen_tokens: int = 0
        self._max_cache_len: int = shape[-2]
        self.k_cache: torch.Tensor = torch.zeros(shape, dtype=dtype, device=device)
        self.v_cache: torch.Tensor = torch.zeros(shape, dtype=dtype, device=device)

    def update(
        self,
        key_states: torch.Tensor,
        value_states: torch.Tensor,
        layer_idx: int,
        *args: Any,
        **kwargs: Any,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Update key/value cache tensors for slice [begin, end).

        Current Transformers Llama calls:
            past_key_values.update(key_states, value_states, layer_idx)

        It does not pass cache_position anymore.
        """
        begin = self.past_seen_tokens
        end = begin + key_states.shape[-2]
        self.k_cache[layer_idx, :, : key_states.shape[1], begin:end, :] = key_states
        self.v_cache[layer_idx, :, : value_states.shape[1], begin:end, :] = value_states

        return (
            self.k_cache[layer_idx, :, :, :end, :],
            self.v_cache[layer_idx, :, :, :end, :],
        )

    def get_seq_length(self, layer_idx: int | None = 0) -> int:
        return self.past_seen_tokens

    def get_max_cache_shape(self, layer_idx: int | None = 0) -> int:
        return self.max_cache_len

    def get_mask_sizes(self, query_length: int, layer_idx: int) -> tuple[int, int]:
        kv_length = self.past_seen_tokens + query_length
        kv_offset = 0
        return kv_length, kv_offset

    @property
    def max_cache_len(self) -> int:
        return self._max_cache_len

    @property
    def is_compileable(self) -> bool:
        return False

    @property
    def is_sliding(self) -> list[bool]:
        return [False] * self.k_cache.shape[0]

In [15]:
class StatefulModelForCausalLM(torch.nn.Module):
    """Model wrapper to swap cache implementation and register as buffers."""

    def __init__(self, model_path: str, max_context_size: int = 4096, batch_size: int = 1) -> None:
        super().__init__()

        self.model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float32)
        config = self.model.config
        self.kv_cache_shape: tuple[int, ...] = (
            config.num_hidden_layers,
            batch_size,
            config.num_key_value_heads,
            max_context_size,
            config.hidden_size // config.num_attention_heads,
        )
        # Register KV cache buffers to be recognized as Core ML states
        self.kv_cache = SliceUpdateKeyValueCache(shape=self.kv_cache_shape)
        self.register_buffer("keyCache", self.kv_cache.k_cache)
        self.register_buffer("valueCache", self.kv_cache.v_cache)

    @torch.no_grad()
    def forward(
        self,
        input_ids: torch.LongTensor,
        causal_mask: torch.Tensor,
    ) -> torch.Tensor:
        # Compute past seen tokens used for updating key/value cache slices
        self.kv_cache.past_seen_tokens = causal_mask.shape[-1] - input_ids.shape[-1]
        return self.model(
            input_ids,
            attention_mask=causal_mask,
            past_key_values=self.kv_cache,
            use_cache=True,
        ).logits

In [16]:
def sample(logits: np.ndarray) -> int:
    """Perform greedy decoding on the logits array to get the next token."""
    return int(np.argmax(logits[0][-1], axis=-1))

def inference(model: ct.models.MLModel, input_ids: np.ndarray, num_past_tokens: int, kv_cache_state) -> np.ndarray:
    """Perform inference with the given model and input data."""
    causal_mask: np.ndarray = np.triu(
        np.full(
            (1, 1, input_ids.shape[-1], num_past_tokens + input_ids.shape[-1]),
            fill_value=-np.inf if num_past_tokens == 0 else 0,
        ),
        k=1,
    ).astype(np.float16)
    outputs: dict[str, np.ndarray] = model.predict(
        data={"inputIds": input_ids, "causalMask": causal_mask},
        state=kv_cache_state,
    )
    return outputs["logits"]

def get_next_token(model: ct.models.MLModel, prompt_tokens: np.ndarray) -> Generator[int, None, None]:
    """Generate a sequence of tokens with naive greedy decoding."""

    kv_cache_state = model.make_state()
    logits: np.ndarray = inference(
        model, 
        input_ids=prompt_tokens, 
        num_past_tokens=0, 
        kv_cache_state=kv_cache_state
    )
    token: int = sample(logits=logits)
    num_past_tokens: int = prompt_tokens.shape[-1]

    while True:
        yield token
        logits: np.ndarray = inference(
            model,
            input_ids=np.array([[token]], dtype=np.int32),
            num_past_tokens=num_past_tokens,
            kv_cache_state=kv_cache_state,
        )
        token: int = sample(logits=logits)
        num_past_tokens += 1

def generate(
    model: ct.models.MLModel,
    prompt: str,
    tokenizer: PreTrainedTokenizer,
    max_new_tokens: int,
) -> str:
    messages = [
        {
            "role": "user", 
            "content": prompt,
        }
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    prefill_start = perf_counter()
    prompt_tokens: np.ndarray = tokenizer(formatted_prompt, return_tensors="np").input_ids
    extend_tokens: list[int] = []
    for i, token in enumerate(get_next_token(model, prompt_tokens=prompt_tokens.astype(np.int32))):
        extend_tokens.append(token)
        if i == 0:
            prefill_end = perf_counter()
            decode_start = prefill_end
            ttft = (prefill_end - prefill_start) * 1000
            print(f"Time to first token: {ttft:.2f} seconds")
        if token == tokenizer.eos_token_id or i + 1 == max_new_tokens:
            decode_end = perf_counter()
            decode_tps = i / (decode_end - decode_start)
            print(f"decode throughput: {decode_tps:.2f} tok/s")
            break
    return tokenizer.decode(prompt_tokens[0].tolist() + extend_tokens)

In [17]:
batch_size, context_size = 1, 2048

In [18]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
device = 'cpu'

# Initialize and trace PyTorch model
torch_model: torch.nn.Module = StatefulModelForCausalLM(
    model_id, batch_size=batch_size, max_context_size=context_size
).eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights: 100%|██████████| 146/146 [00:05<00:00, 27.57it/s]


In [19]:
example_inputs: tuple[torch.Tensor, ...] = (
    torch.zeros((1, 2), dtype=torch.int32, device=device),
    torch.zeros((1, 1, 2, 5), dtype=torch.float32, device=device),
)
traced_model: torch.jit.ScriptModule = torch.jit.trace(
    torch_model.eval(), example_inputs=example_inputs
)

/Users/Sina.Sajadmanesh/Projects/swift-transformers/Examples/Mistral7B/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = query.shape[2] > 1 and attention_mask is None and is_causal


In [20]:
# Convert to Core ML
query_size = ct.RangeDim(lower_bound=1, upper_bound=context_size, default=1)
final_step = ct.RangeDim(lower_bound=1, upper_bound=context_size, default=1)
inputs: list[ct.TensorType] = [
    ct.TensorType(shape=(batch_size, query_size), dtype=np.int32, name="inputIds"),
    ct.TensorType(
        shape=(batch_size, 1, query_size, final_step),
        dtype=np.float16,
        name="causalMask",
    ),
]
states: list[ct.StateType] = [
    ct.StateType(
        wrapped_type=ct.TensorType(shape=torch_model.kv_cache_shape, dtype=np.float16),
        name="keyCache",
    ),
    ct.StateType(
        wrapped_type=ct.TensorType(shape=torch_model.kv_cache_shape, dtype=np.float16),
        name="valueCache",
    ),
]
outputs: list[ct.TensorType] = [ct.TensorType(dtype=np.float16, name="logits")]
mlmodel: ct.models.MLModel = ct.convert(
    traced_model,
    inputs=inputs,
    outputs=outputs,
    states=states,
    minimum_deployment_target=ct.target.macOS15,
    compute_units=ct.ComputeUnit.ALL,
    compute_precision=ct.precision.FLOAT16,
)
mlmodel.save(f'models/{model_id}.mlpackage')

Torch var valueCache is added again.
Torch var keyCache is added again.
Running MIL default pipeline:  67%|██████▋   | 64/95 [00:30<00:36,  1.17s/ passes]/Users/Sina.Sajadmanesh/Projects/swift-transformers/Examples/Mistral7B/.venv/lib/python3.12/site-packages/coremltools/converters/mil/mil/passes/defs/optimize_repeat_ops.py:433: RuntimeWarning: overflow encountered in cast
  max(cur_range.low, tmp_range.low), min(cur_range.high, tmp_range.high)
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 63.47 passes/s]


In [22]:
compiled_model_path = mlmodel.get_compiled_model_path()
copytree(compiled_model_path, f'models/{model_id}.mlmodelc', dirs_exist_ok=True)
mlmodel = ct.models.CompiledMLModel(f'models/{model_id}.mlmodelc')

In [24]:
generate(
    mlmodel,
    prompt="Tell me a true short story about France.",
    tokenizer=tokenizer,
    max_new_tokens=128,
)

Time to first token: 343.91 seconds
decode throughput: 19.75 tok/s


'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 28 May 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell me a true short story about France.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nIt was a chilly winter morning in the French countryside, and the snow-covered fields of Provence stretched out as far as the eye could see. The village of Saint-Pierre, nestled in the heart of the Luberon region, was bustling with activity as the locals prepared for the annual Fête de la Musique, a celebration of music and community.\n\nIn the town square, a young musician named Sophie had set up her guitar and began to play a lively tune on the street corner. The music was infectious, and soon, people of all ages gathered around her, tapping their feet and swaying to the rhythm.\n\nAs the sun rose'